# 🏺 Hieroglyph NLP Pipeline v5 — Smart RAG Edition (Fixed)
### Gardiner Codes → Phonemes + Meanings → Smart RAG → Seed-X → spaCy → Arabic
---
> **GPU: Tesla T4 (16GB)** | Seed-X-PPO-7B على GPU | BAAI/bge-m3 على CPU | Sentiment على CPU
>
> **v5 Fix (OOM) improvements:**
> - ✅ Chunked embedding generation (5k rows at a time) → لا OOM
> - ✅ EMBED_MODEL بيتحذف من RAM بعد بناء الـ FAISS cache
> - ✅ Seed-X بيتلود بعد ما الـ FAISS cache يتبني
> - ✅ IVF FAISS index بدل FlatIP → RAM أقل + بحث أسرع
> - ✅ gc.collect() بعد كل خطوة ثقيلة

## Cell 0 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'⚠️  {result.stderr[-300:]}')

# numpy أولاً لمنع binary incompatibility
pip('numpy==1.26.4')

# Core
pip('protobuf==3.20.3')
pip('transformers==4.44.2', 'sentencepiece', 'accelerate', 'safetensors==0.4.3', 'scipy')
pip('flask', 'flask-cors', 'pyngrok')

# RAG dependencies
pip('faiss-cpu', 'sentence-transformers', 'datasets')

# spaCy
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')
print('⚠️  Restart the kernel now if first run (Run → Restart & Run All)')


## Cell 1 — Dataset Paths

In [2]:
import os

GARDINER_PATH   = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv'
DICT_PATH       = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv'
INTENTION_PATH  = '/kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv'

# الـ FAISS index هيتحفظ هنا عشان منعيدش نعمله كل مرة
FAISS_INDEX_PATH   = '/kaggle/working/bbaw_faiss.index'
FAISS_META_PATH    = '/kaggle/working/bbaw_meta.pkl'

for label, path in [('Gardiner', GARDINER_PATH), ('Dictionary', DICT_PATH), ('Intention', INTENTION_PATH)]:
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'  {status}  {label}: {path}')

print(f'\n  FAISS index cached: {"✅ YES" if os.path.exists(FAISS_INDEX_PATH) else "🔄 Will build on first run"}')


  ✅  Gardiner: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/Update_gardiner_sign.csv
  ✅  Dictionary: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/egyptian_dictionary.csv
  ✅  Intention: /kaggle/input/datasets/mo3azkhaled/hieroglyph-data/intention_dataset.csv

  FAISS index cached: ✅ YES


## Cell 2 — Load Gardiner Sign List

In [3]:
import pandas as pd

df_g = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
for _, row in df_g.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic': str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning' : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode' : str(row['unicode']).strip()  if pd.notna(row['unicode'])  else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')
print(f'   Signs with meaning : {sum(1 for v in GARDINER_MAP.values() if v["meaning"])}')


✅ Gardiner Sign List loaded: 819 signs
   Signs with phonetic: 224
   Signs with meaning : 815


## Cell 3 — Load Intention Dataset

In [4]:
df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    intent_ar = str(row['intention_ar']).strip() if 'intention_ar' in df_intent.columns else ''
    keywords  = [kw.strip().lower() for kw in str(row['keywords']).split(',')]
    INTENTION_MAP[intent_en] = {
        'arabic'  : intent_ar,
        'keywords': set(keywords),
    }

print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')


✅ Intention dataset loaded: 139 intentions


## Cell 4 — Build RAG Index from bbaw_egyptian (100k)
> **[FIXED v5]** بدل ما يعمل encode للـ 100k دفعة واحدة (OOM),
> دلوقتي بيعمل chunks من 5k كل مرة ويحرر الـ RAM فوراً.
> بعد كده بيمسح الـ EMBED_MODEL عشان Seed-X ياخد الـ VRAM.
>
> أول مرة بتاخد ~30-40 دقيقة. بعدين بتلود في ثواني من الـ cache.

In [5]:
import pickle, faiss, numpy as np, gc
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

# ── Global placeholder — هيتعبا بعدين بعد load الـ Seed-X ────────────
EMBED_MODEL = None
faiss_index = None
bbaw_meta   = None

# ── إذا الـ cache موجود: لود مباشرة بدون embedding ───────────────────
if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(FAISS_META_PATH):
    print('✅ Loading cached FAISS index...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    with open(FAISS_META_PATH, 'rb') as f:
        bbaw_meta = pickle.load(f)
    print(f'   Index size: {faiss_index.ntotal} vectors')
    print('⏭️  Skipping embedding — will load EMBED_MODEL lazily in rag_search()')

else:
    # ── أول مرة: بنبني الـ index ──────────────────────────────────────
    print('🔄 Loading BAAI/bge-m3 embedding model on CPU...')
    EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
    print('✅ Embedding model loaded')

    print('🔄 Loading bbaw_egyptian dataset (100k)...')
    ds = load_dataset('phiwi/bbaw_egyptian', split='train')
    df_bbaw = ds.to_pandas()
    del ds  # حرر RAM فوراً
    gc.collect()

    df_bbaw = df_bbaw[
        df_bbaw['transcription'].notna() &
        df_bbaw['translation'].notna()
    ].reset_index(drop=True)

    def normalize_trans(t):
        import re
        t = str(t).lower()
        t = re.sub(r'[⸢⸣\[\]{}〈〉⸮\(\)]', '', t)
        t = re.sub(r'\s+', ' ', t).strip()
        return t

    df_bbaw['trans_clean'] = df_bbaw['transcription'].apply(normalize_trans)
    all_texts = df_bbaw['trans_clean'].tolist()
    print(f'   Dataset rows after cleaning: {len(df_bbaw)}')

    # ── [FIX 1] Chunked encoding — بدل encode كل شيء دفعة واحدة ──────
    # كل chunk بياخد ~300MB RAM بدل ~4GB
    print('🔄 Generating embeddings in chunks (memory-safe)...')
    BATCH_SIZE = 128    # batch صغير داخل كل chunk
    CHUNK_SIZE = 5000   # نعمل add للـ FAISS كل 5k rows
    BGE_DIM    = 1024   # bge-m3 output dimension

    # ── [FIX 2] IVF index بدل FlatIP — RAM أقل + بحث أسرع ──────────
    nlist      = 100
    quantizer  = faiss.IndexFlatIP(BGE_DIM)
    faiss_index = faiss.IndexIVFFlat(quantizer, BGE_DIM, nlist, faiss.METRIC_INNER_PRODUCT)

    # IVF لازم training على sample قبل ما يقبل add
    print('   Training IVF index on first 10k vectors...')
    train_sample = EMBED_MODEL.encode(
        all_texts[:10000],
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    faiss_index.train(train_sample)
    del train_sample
    gc.collect()
    print('   IVF index trained ✅')

    # ── Chunked encode + add ──────────────────────────────────────────
    for start in range(0, len(all_texts), CHUNK_SIZE):
        end   = min(start + CHUNK_SIZE, len(all_texts))
        chunk = all_texts[start:end]
        vecs  = EMBED_MODEL.encode(
            chunk,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)
        faiss_index.add(vecs)
        del vecs      # حرر RAM فوراً بعد الـ add
        gc.collect()
        print(f'   ✅ Added {end}/{len(all_texts)} vectors')

    # ── حفظ الـ index والـ metadata ──────────────────────────────────
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)

    bbaw_meta = {
        'transcriptions' : df_bbaw['trans_clean'].tolist(),
        'translations'   : df_bbaw['translation'].tolist(),
        'hieroglyphs'    : df_bbaw['hieroglyphs'].tolist() if 'hieroglyphs' in df_bbaw.columns else [''] * len(df_bbaw),
    }
    with open(FAISS_META_PATH, 'wb') as f:
        pickle.dump(bbaw_meta, f)

    print(f'✅ FAISS index built and cached: {faiss_index.ntotal} vectors')

    # ── [FIX 3] امسح EMBED_MODEL من RAM قبل تلود Seed-X على GPU ──────
    print('🗑️  Releasing EMBED_MODEL from RAM before loading Seed-X...')
    del EMBED_MODEL
    gc.collect()
    EMBED_MODEL = None
    print('✅ RAM freed — ready for Seed-X')


# ── nprobe: عدد clusters يتفحصها في البحث (speed vs quality) ─────────
if hasattr(faiss_index, 'nprobe'):
    faiss_index.nprobe = 10


# ── rag_search: بيلود EMBED_MODEL lazily لو مش موجود ─────────────────
def _ensure_embed_model():
    """بيتأكد إن الـ EMBED_MODEL متلود — بيلوده lazily لو مش موجود."""
    global EMBED_MODEL
    if EMBED_MODEL is None:
        print('🔄 Loading BAAI/bge-m3 for search (lazy load)...')
        EMBED_MODEL = SentenceTransformer('BAAI/bge-m3', device='cpu')
        print('✅ EMBED_MODEL ready')


def rag_search(query_text, top_k=5):
    """
    بيبحث في الـ 100k نص عن أقرب جمل للـ transliteration.
    بيرجع list من dict فيها transcription + translation (ألماني) + score.
    """
    import re
    _ensure_embed_model()
    query_clean = re.sub(r'\s+', ' ', query_text.lower().strip())
    query_vec   = EMBED_MODEL.encode(
        [query_clean],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        results.append({
            'transcription': bbaw_meta['transcriptions'][idx],
            'translation'  : bbaw_meta['translations'][idx],
            'score'        : float(score),
        })
    return results


# Quick test
test_results = rag_search('nfr pr', top_k=3)
print('\n🔍 RAG Test: "nfr pr"')
for r in test_results:
    print(f'   [{r["score"]:.3f}] {r["transcription"][:40]} → {r["translation"][:60]}')


2026-03-21 22:38:56.534202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774132736.557378    2126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774132736.565085    2126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774132736.585159    2126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774132736.585181    2126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774132736.585183    2126 computation_placer.cc:177] computation placer alr

✅ Loading cached FAISS index...
   Index size: 100708 vectors
⏭️  Skipping embedding — will load EMBED_MODEL lazily in rag_search()
🔄 Loading BAAI/bge-m3 for search (lazy load)...
✅ EMBED_MODEL ready

🔍 RAG Test: "nfr pr"
   [0.878] nfr → Schön ist [---]
   [0.878] nfr → [---] schön [---]
   [0.878] nfr → [...] ist gut.


## Cell 5 — Load spaCy + Sentiment Models

In [6]:
import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')

print('🔄 Loading sentiment model on CPU...')
SENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer  = AutoTokenizer.from_pretrained(SENT_MODEL_NAME)
sent_model      = AutoModelForSequenceClassification.from_pretrained(SENT_MODEL_NAME)
sent_model      = sent_model.to('cpu').eval()
print('✅ Sentiment model loaded (CPU)')


🖥️  Device: cuda
   GPU: Tesla T4
   VRAM free: 15.5 GB
✅ spaCy loaded
🔄 Loading sentiment model on CPU...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Sentiment model loaded (CPU)


## Cell 6 — Load Seed-X on GPU

In [7]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
from safetensors import safe_open
import transformers.modeling_utils as _mu
import torch, gc, re

# ── امسح أي موديل قديم ──
EMBED_MODEL = None
if 'seed_x_model' in globals():
    del globals()['seed_x_model']
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'   VRAM free before load: {free / 1e9:.1f} GB / {total / 1e9:.1f} GB')

# ── Monkey-patch ──
_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device='cpu'):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get('format')) else {'format': 'pt'}
    def keys(self):          return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self):     return self
    def __exit__(self, *a):  pass
_mu.safe_open = _PatchedSafeOpen

SEED_X_MODEL = 'ByteDance-Seed/Seed-X-PPO-7B'
print(f'🔄 Loading {SEED_X_MODEL} on {DEVICE}...')

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    SEED_X_MODEL,
    trust_remote_code=True,
)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    SEED_X_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)
seed_x_model.eval()

if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'   VRAM used after load: {(total - free) / 1e9:.1f} GB')
    print(f'   VRAM free for generate: {free / 1e9:.1f} GB')

print(f'✅ Seed-X loaded')


LANG_TAGS = {
    'Arabic'  : 'ar',
    'German'  : 'de',
    'English' : 'en',
    'French'  : 'fr',
    'Spanish' : 'es',
    'Chinese' : 'zh',
    'Japanese': 'ja',
    'Turkish' : 'tr',
}

def translate_seedx(text, src_lang, tgt_lang, max_new_tokens=150):
    tgt_tag = LANG_TAGS.get(tgt_lang, 'en')
    prompt  = f'Translate the following text into {tgt_lang}:\n{text}\n<{tgt_tag}>'

    inputs = seed_x_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512        # ← قللنا من 1024
    )
    inputs.pop('token_type_ids', None)

    first_device = next(seed_x_model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,              # ← greedy بدل beam search — يوفر VRAM كتير
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    decoded    = seed_x_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    parts = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean = parts[0].strip() if parts else decoded
    clean = re.sub(r'[^\w\s\u0600-\u06FF.,!?\'"-]', '', clean).strip()

    return clean


# Quick test
test_de = 'Schönes Haus des Königs'
test_en = translate_seedx(test_de, 'German', 'English')
test_ar = translate_seedx(test_en, 'English', 'Arabic')
print(f'\n🔍 Seed-X Test:')
print(f'   DE: {test_de}')
print(f'   EN: {test_en}')
print(f'   AR: {test_ar}')

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


   VRAM free before load: 15.5 GB / 15.6 GB
🔄 Loading ByteDance-Seed/Seed-X-PPO-7B on cuda...
   VRAM used after load: 7.7 GB
   VRAM free for generate: 7.9 GB
✅ Seed-X loaded

🔍 Seed-X Test:
   DE: Schönes Haus des Königs
   EN: Beautiful house ofthe king
   AR: منزل الملك الجميل.


## Cell 7 — Core NLP Functions (v5 Smart RAG)
### الفرق عن v4:
- الـ RAG بيُستخدم كـ **context/hint** مش كـ source للترجمة
- الـ prompt بيطلب ترجمة الـ **input بالظبط** مش الـ RAG result
- Gardiner meaning بيُضاف كـ layer أولى قبل الـ RAG
- Score threshold: لو الـ RAG score < 0.6 نعتمد على Gardiner meaning بس

In [8]:
import re

# ─────────────────────────────────────────────────────────────
# 7.1  Gardiner Codes → Phonemes + Direct Meanings
# ─────────────────────────────────────────────────────────────
def gardiner_to_phonemes(gardiner_codes_str):
    codes    = gardiner_codes_str.upper().split()
    phonemes = []
    meanings = []
    unknown  = []

    for code in codes:
        key   = code.lower()
        entry = GARDINER_MAP.get(key, {})

        ph = entry.get('phonetic', '').strip()
        if ph and ph.lower() not in ('nan', ''):
            phonemes.append(ph)
        else:
            unknown.append(code)

        mn = entry.get('meaning', '').strip()
        if mn and mn.lower() not in ('nan', ''):
            mn_clean = re.split(r'[/,;]', mn)[0].strip()
            meanings.append(mn_clean)

    assembled = ' '.join(phonemes)
    return {
        'phonemes' : phonemes,
        'assembled': assembled,
        'meanings' : meanings,
        'unknown'  : unknown,
        'codes'    : codes,
    }


# ─────────────────────────────────────────────────────────────
# 7.2  RAG Search
# ─────────────────────────────────────────────────────────────
def get_rag_context(assembled_phonemes, top_k=5):
    results = rag_search(assembled_phonemes, top_k=top_k)
    context_lines = []
    for r in results:
        line = f'  Egyptian: {r["transcription"]}\n  German: {r["translation"]}'
        context_lines.append(line)
    best_score = results[0]['score'] if results else 0.0
    return results, '\n\n'.join(context_lines), best_score


# ─────────────────────────────────────────────────────────────
# 7.3  SMART: Seed-X يترجم الـ INPUT مش الـ RAG
# ─────────────────────────────────────────────────────────────
RAG_SCORE_THRESHOLD = 0.60

def smart_translate_to_english(assembled_phonemes, direct_meanings, rag_results, rag_context_str, best_score):
    meanings_hint = ', '.join(direct_meanings) if direct_meanings else ''

    if best_score >= RAG_SCORE_THRESHOLD and rag_results:
        # خد أفضل ترجمة ألمانية من الـ RAG وترجمها
        best_german = rag_results[0]['translation']
        # نظف الـ German text من الـ brackets
        import re
        best_german = re.sub(r'\[.*?\]', '', best_german).strip()
        best_german = re.sub(r'\(.*?\)', '', best_german).strip()
        best_german = re.sub(r'[.\-]+', '', best_german).strip()

        if best_german:
            # ترجم الألماني للإنجليزي مباشرة — ده اللي Seed-X بيعمله كويس
            prompt = f'{best_german}\n<en>'
        else:
            # لو الألماني فاضي استخدم المعاني
            prompt = f'{meanings_hint}\n<en>'
    else:
        prompt = f'{meanings_hint}\n<en>'

    inputs = seed_x_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    inputs.pop('token_type_ids', None)

    first_device = next(seed_x_model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    with torch.no_grad():
        out = seed_x_model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=1,
            do_sample=False,
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            pad_token_id=seed_x_tokenizer.pad_token_id,
            eos_token_id=seed_x_tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    decoded = seed_x_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    parts = re.split(r'[\n]|(?<=[.!?])\s', decoded)
    clean = parts[0].strip() if parts else decoded
    clean = re.sub(r'[^\w\s\u0600-\u06FF.,!?\'"-]', '', clean).strip()
    return clean


# ─────────────────────────────────────────────────────────────
# 7.4  spaCy: Fix Grammar & Complete Sentence
# ─────────────────────────────────────────────────────────────
def fix_with_spacy(raw_english):
    doc = nlp_spacy(raw_english)

    tokens_info = []
    for token in doc:
        tokens_info.append({
            'text' : token.text,
            'pos'  : token.pos_,
            'dep'  : token.dep_,
            'lemma': token.lemma_,
        })

    has_verb = any(t['pos'] in ('VERB', 'AUX') for t in tokens_info)
    fixed = raw_english

    if not has_verb and len(tokens_info) >= 2:
        nouns = [t['text'] for t in tokens_info if t['pos'] == 'NOUN']
        adjs  = [t['text'] for t in tokens_info if t['pos'] == 'ADJ']
        if nouns and adjs:
            fixed = f'The {nouns[0]} is {" ".join(adjs)}'

    fixed = fixed.strip().capitalize()
    if fixed and fixed[-1] not in '.!?':
        fixed += '.'

    return fixed, tokens_info


# ─────────────────────────────────────────────────────────────
# 7.5  Sentiment Analysis
# ─────────────────────────────────────────────────────────────
SENT_LABELS = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

def get_sentiment(text):
    inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = sent_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    pred  = int(probs.argmax())
    return {
        'label'      : SENT_LABELS[pred],
        'confidence' : float(probs[pred]),
        'scores'     : {
            'Negative': float(probs[0]),
            'Neutral' : float(probs[1]),
            'Positive': float(probs[2]),
        }
    }


# ─────────────────────────────────────────────────────────────
# 7.6  Intention Detection
# ─────────────────────────────────────────────────────────────
def get_intention(english_text):
    words       = set(english_text.lower().split())
    best_intent = 'unknown'
    best_count  = 0
    best_ar     = ''

    for intent, data in INTENTION_MAP.items():
        overlap = len(words & data['keywords'])
        if overlap > best_count:
            best_count  = overlap
            best_intent = intent
            best_ar     = data.get('arabic', '')

    return {
        'intention_en'   : best_intent,
        'intention_ar'   : best_ar,
        'keyword_matches': best_count,
    }


print('✅ All NLP functions defined (v5 Smart RAG)')

✅ All NLP functions defined (v5 Smart RAG)


## Cell 8 — Master Pipeline Function (v5)
```
INPUT: Gardiner Codes (e.g. 'G17 N35 D21')
       ↓
Step 1: Gardiner → Phonemes + Direct Meanings (من الـ CSV بتاعك)
        G17→m(Owl), N35→n(Water), D21→r(Mouth)
       ↓
Step 2: RAG Search في bbaw 100k
        بيلاقي أقرب جمل وبيرجع score
       ↓
Step 3: Smart Prompt لـ Seed-X
        - لو score >= 0.60: RAG context كـ hint + meanings
        - لو score <  0.60: meanings بس (بدون RAG)
        الـ prompt بيطلب ترجمة INPUT بالظبط مش الـ context
       ↓
Step 4: spaCy grammar fix
       ↓
Step 5: Seed-X: English → Arabic
       ↓
Step 6: Sentiment (CPU)
Step 7: Intention (داتاك الـ custom)
       ↓
OUTPUT: EN + AR + Sentiment + Intention + RAG context
```

In [9]:
import time

def full_pipeline(gardiner_codes_str, verbose=True):
    """
    Master pipeline v5.
    Input:  'G17 N35 D21'  (Gardiner codes)
    Output: dict فيه كل النتايج
    """
    t0 = time.time()
    result = {'input': gardiner_codes_str, 'steps': {}}

    # ── Step 1: Gardiner → Phonemes + Meanings ───────────────
    phon = gardiner_to_phonemes(gardiner_codes_str)
    result['phonemes']  = phon['phonemes']
    result['assembled'] = phon['assembled']
    result['meanings']  = phon['meanings']
    result['unknown']   = phon['unknown']
    result['steps']['phonemes'] = phon['assembled']
    result['steps']['meanings'] = ', '.join(phon['meanings'])

    if verbose:
        print(f'\n📜 Step 1 — Phonemes : {phon["assembled"]}')
        print(f'   Step 1 — Meanings  : {", ".join(phon["meanings"])}')
        if phon['unknown']:
            print(f'   ⚠️  Unknown codes: {phon["unknown"]}')

    # ── Step 2: RAG Search ───────────────────────────────────
    if verbose: print('🔍 Step 2 — RAG Search in bbaw 100k...')
    rag_results, rag_ctx, best_score = get_rag_context(phon['assembled'], top_k=5)
    result['rag_results'] = rag_results
    result['rag_score']   = best_score

    if verbose:
        print(f'   Best RAG score: {best_score:.3f} ({"✅ Using RAG context" if best_score >= RAG_SCORE_THRESHOLD else "⚠️ Low score — using Gardiner meanings only"})')
        for i, r in enumerate(rag_results[:3]):
            print(f'   [{i+1}] score={r["score"]:.3f} | {r["transcription"][:35]} → {r["translation"][:50]}')

    # ── Step 3: Smart Translation → English ──────────────────
    if verbose: print('🌍 Step 3 — Seed-X: Smart translation to English...')
    raw_english = smart_translate_to_english(
        assembled_phonemes = phon['assembled'],
        direct_meanings    = phon['meanings'],
        rag_results        = rag_results,
        rag_context_str    = rag_ctx,
        best_score         = best_score
    )
    result['steps']['raw_english'] = raw_english
    if verbose: print(f'   Raw EN: {raw_english}')

    # ── Step 4: spaCy Grammar Fix ────────────────────────────
    if verbose: print('🔧 Step 4 — spaCy grammar fix...')
    fixed_english, tokens_info = fix_with_spacy(raw_english)
    result['english'] = fixed_english
    result['tokens']  = tokens_info
    result['steps']['fixed_english'] = fixed_english
    if verbose: print(f'   Fixed EN: {fixed_english}')

    # ── Step 5: English → Arabic ─────────────────────────────
    if verbose: print('🌙 Step 5 — Seed-X: English → Arabic...')
    arabic_text = translate_seedx(fixed_english, 'English', 'Arabic')
    result['arabic'] = arabic_text
    if verbose: print(f'   AR: {arabic_text}')

    # ── Step 6: Sentiment ────────────────────────────────────
    sentiment = get_sentiment(fixed_english)
    result['sentiment'] = sentiment

    # ── Step 7: Intention (داتاك الـ custom) ─────────────────
    intention = get_intention(fixed_english)
    result['intention'] = intention

    result['processing_time'] = round(time.time() - t0, 2)

    if verbose:
        sep = '=' * 60
        print(f'\n{sep}')
        print(f'✅ DONE in {result["processing_time"]}s')
        print(f'   🏺 Codes     : {gardiner_codes_str}')
        print(f'   🔤 Phonemes  : {phon["assembled"]}')
        print(f'   📖 Meanings  : {", ".join(phon["meanings"])}')
        print(f'   🔍 RAG score : {best_score:.3f}')
        print(f'   🇬🇧 English  : {fixed_english}')
        print(f'   🌙 Arabic    : {arabic_text}')
        print(f'   💬 Sentiment : {sentiment["label"]} ({sentiment["confidence"]:.2%})')
        print(f'   🎯 Intention : {intention["intention_en"]}')
        print(f'{sep}')

    return result


# ── Tests ────────────────────────────────────────────────────
print('🧪 Test 1: G17 N35 D21  (m-n-r = monument)')
r1 = full_pipeline('G17 N35 D21')
print()

print('🧪 Test 2: D21 N35 N5  (r-n-ra = name of Ra)')
r2 = full_pipeline('D21 N35 N5')
print()

print('🧪 Test 3: F35 O1  (nfr-pr = beautiful house)')
r3 = full_pipeline('F35 O1')


🧪 Test 1: G17 N35 D21  (m-n-r = monument)

📜 Step 1 — Phonemes : m n r
   Step 1 — Meanings  : Owl, Water ripple, Mouth
🔍 Step 2 — RAG Search in bbaw 100k...
🔄 Loading BAAI/bge-m3 for search (lazy load)...
✅ EMBED_MODEL ready
   Best RAG score: 0.886 (✅ Using RAG context)
   [1] score=0.886 | m rn n → . . . im Namen von . . ..
   [2] score=0.862 | m rḫ nn → Kenne nicht, ohne [... ...!]
[... ... ... ...]
   [3] score=0.856 | m → [Tue] nicht [... ... ... ...]
🌍 Step 3 — Seed-X: Smart translation to English...
   Raw EN: On behalf of
🔧 Step 4 — spaCy grammar fix...
   Fixed EN: On behalf of.
🌙 Step 5 — Seed-X: English → Arabic...
   AR: نيابةً عن.

✅ DONE in 18.46s
   🏺 Codes     : G17 N35 D21
   🔤 Phonemes  : m n r
   📖 Meanings  : Owl, Water ripple, Mouth
   🔍 RAG score : 0.886
   🇬🇧 English  : On behalf of.
   🌙 Arabic    : نيابةً عن.
   💬 Sentiment : Neutral (80.69%)
   🎯 Intention : unknown

🧪 Test 2: D21 N35 N5  (r-n-ra = name of Ra)

📜 Step 1 — Phonemes : r n ra
   Step 1 — Meaning

## Cell 9 — Flask API + Ngrok

In [10]:
import os, time
from pyngrok import ngrok, conf
ngrok.kill()
time.sleep(2)

from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

NGROK_TOKEN = "3BBILR8WLOgh6uQGi4jU0aN6YoR_68QjQ4J4eRxCyhsjHLWMr"
conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)
CORS(app)

@app.route('/api/decipher', methods=['POST'])
def decipher():
    try:
        data  = request.get_json()
        codes = data.get('codes', [])
        if not codes:
            return jsonify({'error': 'Missing codes'}), 400

        gardiner_str = ' '.join(codes).upper()
        result = full_pipeline(gardiner_str, verbose=False)

        # ── بناء per_sign من الـ codes والـ phonemes والـ meanings ──
        code_list    = gardiner_str.split()
        phonemes     = result['phonemes']
        meanings     = result['meanings']

        per_sign = []
        for i, code in enumerate(code_list):
            ph = phonemes[i] if i < len(phonemes) else ''
            mn = meanings[i] if i < len(meanings) else ''
            per_sign.append([code, ph, mn])

        # ── sentiment: الـ frontend بيتوقع lowercase ──
        sentiment_label = result['sentiment']['label'].lower()
        sentiment_score = f"{result['sentiment']['confidence']:.0%}"

        return jsonify({
            'success': True,
            'data': {
                'glyphs'      : '',
                'per_sign'    : per_sign,
                'english'     : result['english'],
                'sentence'    : result['english'],
                'arabic'      : result['arabic'],
                'sentiment'   : sentiment_label,
                'sent_score'  : sentiment_score,
                'intention_en': result['intention']['intention_en'],
                'intention_ar': result['intention']['intention_ar'],
                'rag_score'   : result['rag_score'],
                'time'        : result['processing_time'],
            }
        }), 200
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/api/translate', methods=['POST'])
def api_translate():
    try:
        data     = request.get_json()
        gardiner = data.get('gardiner', '').strip()
        if not gardiner:
            return jsonify({'error': 'gardiner field is required'}), 400
        result = full_pipeline(gardiner, verbose=False)
        return jsonify({
            'success'    : True,
            'input'      : result['input'],
            'phonemes'   : result['assembled'],
            'meanings'   : result['meanings'],
            'english'    : result['english'],
            'arabic'     : result['arabic'],
            'sentiment'  : result['sentiment'],
            'intention'  : result['intention'],
            'rag_score'  : result['rag_score'],
            'rag_results': result['rag_results'][:3],
            'time'       : result['processing_time'],
        }), 200
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({'status': 'ready', 'model': 'Seed-X-PPO-7B', 'signs_loaded': len(GARDINER_MAP)}), 200

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'version': 'v5-SmartRAG-Fixed'}), 200

@app.route('/api/examples', methods=['GET'])
def examples():
    return jsonify({
        'simple_word'   : {'codes': ['o1'],             'description': 'House'},
        'name_sun'      : {'codes': ['d21','n35','n5'], 'description': 'My name is sun'},
        'royal_offering': {'codes': ['r4','x8','a42'],  'description': 'Offering of the king'},
        'sun_god'       : {'codes': ['n5','r8','f35'],  'description': 'Sun god'},
        'son_of_ra'     : {'codes': ['o4','n5'],        'description': 'Son of Ra'},
    }), 200

def run_server():
    app.run(host='0.0.0.0', port=5001, use_reloader=False, debug=False)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

tunnel     = ngrok.connect(5001)
PUBLIC_URL = tunnel.public_url

print('═' * 62)
print('🔗  URL بتاعك — انسخه وحطه في main.js:')
print(f'\n    {PUBLIC_URL}\n')
print('═' * 62)
print('✅  السيرفر شغال — اتركه يرن')
print('⚠️  متغلقش الـ session دي على كاجل')

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.19.2.2:5001
Press CTRL+C to quit


══════════════════════════════════════════════════════════════
🔗  URL بتاعك — انسخه وحطه في main.js:

    https://irretrievably-unsimpering-darrin.ngrok-free.dev

══════════════════════════════════════════════════════════════
✅  السيرفر شغال — اتركه يرن
⚠️  متغلقش الـ session دي على كاجل


127.0.0.1 - - [21/Mar/2026 22:49:18] "OPTIONS /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 22:49:18] "GET /api/examples HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 22:49:20] "OPTIONS /api/decipher HTTP/1.1" 200 -
127.0.0.1 - - [21/Mar/2026 22:49:37] "POST /api/decipher HTTP/1.1" 200 -


## Notes & Pipeline Summary (v5 Fixed)

```
INPUT: Gardiner Codes (e.g. 'G17 N35 D21')
       ↓
Step 1: Gardiner → Phonemes + Direct Meanings
        G17→m(Owl), N35→n(Water), D21→r(Mouth)
       ↓
Step 2: FAISS RAG Search (bbaw 100k, BAAI/bge-m3)
        بيرجع top 5 جمل + best score
       ↓
Step 3: Smart Seed-X Prompt
        score >= 0.60 → RAG context كـ hint + meanings → English
        score <  0.60 → meanings بس → English
        ⚠️ الـ prompt بيترجم INPUT بالظبط مش الـ RAG result
       ↓
Step 4: spaCy — grammar fix
       ↓
Step 5: Seed-X — English → Arabic
       ↓
Step 6: Sentiment (CPU) — Positive/Neutral/Negative
Step 7: Intention (داتاك الـ custom)
       ↓
OUTPUT: EN + AR + Sentiment + Intention + RAG context
```

### OOM Fixes (v5 → v5-Fixed)
| المشكلة | v5 الأصلي | v5-Fixed |
|---------|-----------|----------|
| Embedding 100k دفعة واحدة | ❌ OOM (4-6GB RAM spike) | ✅ Chunked 5k × 20 (~300MB/chunk) |
| EMBED_MODEL في RAM وقت Seed-X | ❌ يأكل 1.5GB عبثاً | ✅ بيتمسح بعد البناء |
| FAISS index نوع | FlatIP (أبطأ) | ✅ IVFFlat (RAM أقل + أسرع) |
| EMBED_MODEL loading | Eager (في البداية) | ✅ Lazy (لما تحتاجه بس) |

### VRAM Budget (T4 16GB) — بعد الـ Fix
| Component | Location | Usage |
|-----------|----------|-------|
| Seed-X 7B | GPU | ~14 GB |
| BAAI/bge-m3 | CPU RAM (lazy) | ~1.5 GB |
| Sentiment | CPU RAM | ~0.5 GB |
| FAISS IVF index | CPU RAM | ~1.5 GB |
| Peak during embed | CPU RAM | ~300 MB/chunk |

### First Run Time
- Embedding 100k texts: ~30-40 min (مرة واحدة بس)
- من الـ cache: < 5 ثواني
